# 11. Simulation Methods

Simulation uses random sampling to solve problems numerically when analytical solutions (using calculus) are difficult or impossible.

**Why Simulation Matters for Calculus:**
- Monte Carlo integration approximates $\int_a^b f(x)dx$ numerically
- Validates theoretical calculus results empirically
- Solves high-dimensional integrals that are analytically intractable
- Tests statistical theorems like the Central Limit Theorem

**Topics Covered:**
1. Random number generation
2. Monte Carlo integration (numerical approximation of integrals)
3. Statistical simulations (bootstrap, permutation tests)
4. Random walks and stochastic processes
5. Discrete event simulation
6. Applications when calculus fails

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, integrate
import pandas as pd

sns.set_style('whitegrid')
np.random.seed(42)

## 1. Random Number Generation

All simulation starts with generating random numbers from different distributions.

**Inverse Transform Method** (uses calculus!):
If $U \sim \text{Uniform}(0,1)$ and $F$ is a CDF, then $X = F^{-1}(U)$ has CDF $F$.

Remember: $F(x) = \int_{-\infty}^x f(t)dt$ (CDF via integration!)

In [ ]:
# Generate uniform random numbers
uniform_samples = np.random.uniform(0, 1, 1000)

# Inverse transform method for exponential distribution
# For Exponential(λ): F(x) = 1 - e^(-λx)
# Inverse: F^(-1)(u) = -ln(1-u)/λ
lambda_param = 2.0
exponential_samples = -np.log(1 - uniform_samples) / lambda_param

# Compare with numpy's built-in
exponential_builtin = np.random.exponential(1/lambda_param, 1000)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Uniform distribution
axes[0].hist(uniform_samples, bins=30, density=True, alpha=0.7, edgecolor='black')
axes[0].axhline(1, color='red', linestyle='--', linewidth=2, label='Theoretical PDF')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')
axes[0].set_title('Uniform(0,1) Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Exponential via inverse transform
axes[1].hist(exponential_samples, bins=30, density=True, alpha=0.7, edgecolor='black', label='Inverse Transform')
x = np.linspace(0, 4, 100)
axes[1].plot(x, lambda_param * np.exp(-lambda_param * x), 'r--', linewidth=2, label='Theoretical PDF')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Exponential(λ={lambda_param}) via Inverse Transform')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Comparison
axes[2].hist(exponential_builtin, bins=30, density=True, alpha=0.7, edgecolor='black', label='NumPy Built-in')
axes[2].plot(x, lambda_param * np.exp(-lambda_param * x), 'r--', linewidth=2, label='Theoretical PDF')
axes[2].set_xlabel('Value')
axes[2].set_ylabel('Density')
axes[2].set_title('NumPy Exponential (Validation)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Inverse Transform Mean: {exponential_samples.mean():.4f}")
print(f"Theoretical Mean: {1/lambda_param:.4f}")
print(f"NumPy Built-in Mean: {exponential_builtin.mean():.4f}")

## 2. Monte Carlo Integration

**The Big Idea:** Approximate $\int_a^b f(x)dx$ using random sampling.

**Method 1 - Hit or Miss:**
$$\int_a^b f(x)dx \approx (b-a) \cdot c \cdot \frac{\text{# hits}}{\text{# total}}$$

**Method 2 - Mean Value (better!):**
$$\int_a^b f(x)dx \approx (b-a) \cdot \frac{1}{N}\sum_{i=1}^N f(x_i)$$
where $x_i \sim \text{Uniform}(a,b)$

This comes from: $E[f(X)] = \int_a^b f(x) \cdot \frac{1}{b-a}dx$ (expected value via integration!)

In [ ]:
# Example: Estimate π using Monte Carlo
# Circle area = πr² = π (for r=1)
# Square area = (2r)² = 4
# Ratio = π/4, so π ≈ 4 * (points inside circle / total points)

def estimate_pi(n_samples):
    """Estimate π using Monte Carlo hit-or-miss method."""
    x = np.random.uniform(-1, 1, n_samples)
    y = np.random.uniform(-1, 1, n_samples)
    inside_circle = (x**2 + y**2) <= 1
    pi_estimate = 4 * inside_circle.sum() / n_samples
    return pi_estimate, x, y, inside_circle

# Test with increasing sample sizes
sample_sizes = [100, 1000, 10000, 100000]
estimates = []
errors = []

for n in sample_sizes:
    pi_est, _, _, _ = estimate_pi(n)
    estimates.append(pi_est)
    errors.append(abs(pi_est - np.pi))

# Visualize for largest sample
pi_est, x, y, inside = estimate_pi(5000)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot
axes[0].scatter(x[inside], y[inside], c='blue', s=1, alpha=0.5, label='Inside circle')
axes[0].scatter(x[~inside], y[~inside], c='red', s=1, alpha=0.5, label='Outside circle')
theta = np.linspace(0, 2*np.pi, 100)
axes[0].plot(np.cos(theta), np.sin(theta), 'g-', linewidth=2, label='Unit circle')
axes[0].set_xlim(-1.1, 1.1)
axes[0].set_ylim(-1.1, 1.1)
axes[0].set_aspect('equal')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title(f'Monte Carlo Estimation of π\nEstimate: {pi_est:.6f}, True: {np.pi:.6f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Convergence plot
axes[1].semilogx(sample_sizes, estimates, 'bo-', linewidth=2, markersize=8, label='MC Estimates')
axes[1].axhline(np.pi, color='red', linestyle='--', linewidth=2, label='True π')
axes[1].set_xlabel('Number of Samples (log scale)')
axes[1].set_ylabel('π Estimate')
axes[1].set_title('Convergence to True Value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nConvergence Analysis:")
for n, est, err in zip(sample_sizes, estimates, errors):
    print(f"N = {n:6d}: π ≈ {est:.6f}, Error = {err:.6f}")

## 3. Monte Carlo Integration - Mean Value Method

Let's approximate a definite integral and compare with the analytical solution.

**Example:** $\int_0^1 e^{-x^2}dx$ (no closed form, but we can use numerical integration to verify)

In [ ]:
def f(x):
    """Function to integrate: e^(-x²)"""
    return np.exp(-x**2)

# Analytical solution (via scipy quad)
true_value, _ = integrate.quad(f, 0, 1)

# Monte Carlo mean value method
def monte_carlo_integrate(func, a, b, n_samples):
    """Estimate integral using mean value method."""
    x = np.random.uniform(a, b, n_samples)
    return (b - a) * np.mean(func(x))

# Test with increasing samples
sample_sizes = np.logspace(2, 5, 20).astype(int)
mc_estimates = []
mc_errors = []

for n in sample_sizes:
    estimate = monte_carlo_integrate(f, 0, 1, n)
    mc_estimates.append(estimate)
    mc_errors.append(abs(estimate - true_value))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Function visualization
x_plot = np.linspace(0, 1, 200)
axes[0].plot(x_plot, f(x_plot), 'b-', linewidth=2, label='$f(x) = e^{-x^2}$')
axes[0].fill_between(x_plot, 0, f(x_plot), alpha=0.3)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title(f'Integral to Estimate: $\int_0^1 e^{{-x^2}}dx \approx {true_value:.6f}$')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Convergence plot
axes[1].loglog(sample_sizes, mc_errors, 'bo-', linewidth=2, markersize=6, label='MC Error')
# Theoretical O(1/√N) convergence
axes[1].loglog(sample_sizes, 0.5/np.sqrt(sample_sizes), 'r--', linewidth=2, label='$O(1/\sqrt{N})$ reference')
axes[1].set_xlabel('Number of Samples')
axes[1].set_ylabel('Absolute Error')
axes[1].set_title('Monte Carlo Error Convergence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"True value (scipy.quad): {true_value:.8f}")
print(f"MC estimate (N=100,000): {mc_estimates[-1]:.8f}")
print(f"Error: {mc_errors[-1]:.8f}")
print(f"\nNote: MC integration has O(1/√N) convergence regardless of dimension!")

## 4. High-Dimensional Integration

**Where Monte Carlo Shines:** High-dimensional integrals where calculus-based methods fail.

**Example:** Volume of a hypersphere in 5 dimensions
$$V = \int \int \int \int \int_{x_1^2+x_2^2+x_3^2+x_4^2+x_5^2 \leq 1} dx_1 dx_2 dx_3 dx_4 dx_5$$

Analytical: $V_n(r) = \frac{\pi^{n/2}}{\Gamma(n/2 + 1)}r^n$ for n-dimensional sphere

In [ ]:
def hypersphere_volume_mc(n_dim, radius=1.0, n_samples=100000):
    """Estimate volume of n-dimensional hypersphere using Monte Carlo."""
    # Generate random points in n-dimensional hypercube [-r, r]^n
    points = np.random.uniform(-radius, radius, (n_samples, n_dim))
    # Check if inside hypersphere
    distances = np.sqrt(np.sum(points**2, axis=1))
    inside = distances <= radius
    # Volume of hypercube is (2r)^n
    hypercube_volume = (2 * radius) ** n_dim
    # Estimate hypersphere volume
    return hypercube_volume * inside.sum() / n_samples

def hypersphere_volume_analytical(n_dim, radius=1.0):
    """Analytical formula for n-dimensional hypersphere volume."""
    from scipy.special import gamma
    return (np.pi ** (n_dim/2)) / gamma(n_dim/2 + 1) * (radius ** n_dim)

# Test for dimensions 1 through 10
dimensions = range(1, 11)
mc_volumes = []
analytical_volumes = []
errors = []

for d in dimensions:
    mc_vol = hypersphere_volume_mc(d, n_samples=100000)
    analytical_vol = hypersphere_volume_analytical(d)
    mc_volumes.append(mc_vol)
    analytical_volumes.append(analytical_vol)
    errors.append(abs(mc_vol - analytical_vol) / analytical_vol * 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Volume comparison
axes[0].plot(dimensions, analytical_volumes, 'ro-', linewidth=2, markersize=8, label='Analytical')
axes[0].plot(dimensions, mc_volumes, 'bs--', linewidth=2, markersize=8, label='Monte Carlo')
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Volume')
axes[0].set_title('Hypersphere Volume vs Dimension (radius=1)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Relative error
axes[1].bar(dimensions, errors, color='purple', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Relative Error (%)')
axes[1].set_title('Monte Carlo Relative Error')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nHypersphere Volumes (radius=1):")
print("Dim | Analytical  | Monte Carlo | Rel Error")
print("-" * 50)
for d, ana, mc, err in zip(dimensions, analytical_volumes, mc_volumes, errors):
    print(f"{d:3d} | {ana:10.6f} | {mc:10.6f} | {err:6.2f}%")

## 5. Bootstrap Resampling

**Bootstrap** estimates the distribution of a statistic by resampling with replacement.

**Why it works:** Central Limit Theorem! As we resample, the distribution of sample means approaches normal.

**Application:** Estimate confidence intervals without knowing the theoretical distribution.

In [ ]:
# Original sample from unknown distribution
np.random.seed(42)
original_data = np.random.exponential(scale=2.0, size=50)  # Unknown to us in practice

# Bootstrap procedure
n_bootstrap = 10000
bootstrap_means = []
bootstrap_medians = []

for _ in range(n_bootstrap):
    # Resample with replacement
    bootstrap_sample = np.random.choice(original_data, size=len(original_data), replace=True)
    bootstrap_means.append(np.mean(bootstrap_sample))
    bootstrap_medians.append(np.median(bootstrap_sample))

bootstrap_means = np.array(bootstrap_means)
bootstrap_medians = np.array(bootstrap_medians)

# Calculate confidence intervals
ci_mean = np.percentile(bootstrap_means, [2.5, 97.5])
ci_median = np.percentile(bootstrap_medians, [2.5, 97.5])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original data
axes[0, 0].hist(original_data, bins=20, density=True, alpha=0.7, edgecolor='black')
axes[0, 0].axvline(np.mean(original_data), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(original_data):.2f}')
axes[0, 0].axvline(np.median(original_data), color='green', linestyle='--', linewidth=2, label=f'Median = {np.median(original_data):.2f}')
axes[0, 0].set_xlabel('Value')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Original Data (n=50)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Bootstrap distribution of means
axes[0, 1].hist(bootstrap_means, bins=50, density=True, alpha=0.7, edgecolor='black')
axes[0, 1].axvline(np.mean(bootstrap_means), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(bootstrap_means):.2f}')
axes[0, 1].axvline(ci_mean[0], color='blue', linestyle=':', linewidth=2, label=f'95% CI: [{ci_mean[0]:.2f}, {ci_mean[1]:.2f}]')
axes[0, 1].axvline(ci_mean[1], color='blue', linestyle=':', linewidth=2)
axes[0, 1].set_xlabel('Sample Mean')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title('Bootstrap Distribution of Mean')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Bootstrap distribution of medians
axes[1, 0].hist(bootstrap_medians, bins=50, density=True, alpha=0.7, edgecolor='black', color='green')
axes[1, 0].axvline(np.mean(bootstrap_medians), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(bootstrap_medians):.2f}')
axes[1, 0].axvline(ci_median[0], color='blue', linestyle=':', linewidth=2, label=f'95% CI: [{ci_median[0]:.2f}, {ci_median[1]:.2f}]')
axes[1, 0].axvline(ci_median[1], color='blue', linestyle=':', linewidth=2)
axes[1, 0].set_xlabel('Sample Median')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Bootstrap Distribution of Median')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Q-Q plot to check normality
stats.probplot(bootstrap_means, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot: Bootstrap Means vs Normal')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Bootstrap Results:")
print(f"Original sample mean: {np.mean(original_data):.4f}")
print(f"Bootstrap mean estimate: {np.mean(bootstrap_means):.4f}")
print(f"95% CI for mean: [{ci_mean[0]:.4f}, {ci_mean[1]:.4f}]")
print(f"\nOriginal sample median: {np.median(original_data):.4f}")
print(f"Bootstrap median estimate: {np.mean(bootstrap_medians):.4f}")
print(f"95% CI for median: [{ci_median[0]:.4f}, {ci_median[1]:.4f}]")

## 6. Permutation Tests

Test if two groups are different by randomly shuffling labels.

**Null Hypothesis:** The groups are not different (labels don't matter).

**Method:** 
1. Calculate observed test statistic (e.g., difference in means)
2. Randomly permute labels many times
3. Calculate test statistic for each permutation
4. p-value = fraction of permutations with statistic as extreme as observed

In [ ]:
# Two groups with possibly different means
np.random.seed(42)
group_A = np.random.normal(loc=10, scale=2, size=30)
group_B = np.random.normal(loc=11.5, scale=2, size=30)

# Observed difference in means
observed_diff = np.mean(group_B) - np.mean(group_A)

# Permutation test
n_permutations = 10000
permuted_diffs = []

combined = np.concatenate([group_A, group_B])
n_A = len(group_A)

for _ in range(n_permutations):
    # Randomly shuffle combined data
    shuffled = np.random.permutation(combined)
    # Split into two groups
    perm_A = shuffled[:n_A]
    perm_B = shuffled[n_A:]
    # Calculate difference
    permuted_diffs.append(np.mean(perm_B) - np.mean(perm_A))

permuted_diffs = np.array(permuted_diffs)

# Calculate p-value (two-tailed)
p_value = np.mean(np.abs(permuted_diffs) >= np.abs(observed_diff))

# Also run t-test for comparison
t_stat, t_pvalue = stats.ttest_ind(group_B, group_A)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Original data
axes[0].hist(group_A, bins=15, alpha=0.7, label=f'Group A (mean={np.mean(group_A):.2f})', edgecolor='black')
axes[0].hist(group_B, bins=15, alpha=0.7, label=f'Group B (mean={np.mean(group_B):.2f})', edgecolor='black')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'Original Data\nObserved Difference: {observed_diff:.3f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Permutation distribution
axes[1].hist(permuted_diffs, bins=50, density=True, alpha=0.7, edgecolor='black')
axes[1].axvline(observed_diff, color='red', linestyle='--', linewidth=2, label=f'Observed: {observed_diff:.3f}')
axes[1].axvline(-observed_diff, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Difference in Means (B - A)')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Permutation Distribution\np-value = {p_value:.4f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Permutation Test Results:")
print(f"Observed difference (B - A): {observed_diff:.4f}")
print(f"Permutation p-value: {p_value:.4f}")
print(f"\nt-test p-value (for comparison): {t_pvalue:.4f}")
print(f"\nConclusion: {'Reject' if p_value < 0.05 else 'Fail to reject'} null hypothesis at α=0.05")

## 7. Random Walks

A **random walk** is a path consisting of random steps.

**1D Random Walk:** $S_n = \sum_{i=1}^n X_i$ where $X_i \in \{-1, +1\}$ with equal probability

**Connection to Calculus:** Brownian motion (continuous limit of random walk) is fundamental to stochastic calculus!

In [ ]:
# Simulate multiple 1D random walks
n_steps = 1000
n_walks = 5

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1D Random Walks
for _ in range(n_walks):
    steps = np.random.choice([-1, 1], size=n_steps)
    position = np.cumsum(steps)
    axes[0, 0].plot(position, alpha=0.7, linewidth=1.5)

axes[0, 0].set_xlabel('Step')
axes[0, 0].set_ylabel('Position')
axes[0, 0].set_title('1D Random Walks')
axes[0, 0].axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[0, 0].grid(True, alpha=0.3)

# 2D Random Walk
steps_x = np.random.choice([-1, 1], size=n_steps)
steps_y = np.random.choice([-1, 1], size=n_steps)
pos_x = np.cumsum(steps_x)
pos_y = np.cumsum(steps_y)

axes[0, 1].plot(pos_x, pos_y, 'b-', alpha=0.6, linewidth=1)
axes[0, 1].plot(0, 0, 'go', markersize=10, label='Start')
axes[0, 1].plot(pos_x[-1], pos_y[-1], 'ro', markersize=10, label='End')
axes[0, 1].set_xlabel('x position')
axes[0, 1].set_ylabel('y position')
axes[0, 1].set_title('2D Random Walk')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_aspect('equal')

# Distribution of final positions (many walks)
n_simulations = 10000
final_positions = []
for _ in range(n_simulations):
    steps = np.random.choice([-1, 1], size=n_steps)
    final_positions.append(np.sum(steps))

final_positions = np.array(final_positions)

axes[1, 0].hist(final_positions, bins=50, density=True, alpha=0.7, edgecolor='black')
# Theoretical: Normal(0, sqrt(n)) by CLT
x_theory = np.linspace(final_positions.min(), final_positions.max(), 100)
axes[1, 0].plot(x_theory, stats.norm.pdf(x_theory, 0, np.sqrt(n_steps)), 'r-', linewidth=2, label='Normal(0, √n)')
axes[1, 0].set_xlabel('Final Position')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title(f'Distribution of Final Position\n(n={n_steps} steps, {n_simulations} simulations)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Mean squared displacement over time
time_points = np.arange(1, n_steps+1)
msd = []

for t in range(1, n_steps+1, 10):
    displacements = []
    for _ in range(1000):
        steps = np.random.choice([-1, 1], size=t)
        displacements.append(np.sum(steps)**2)
    msd.append(np.mean(displacements))

time_sampled = range(1, n_steps+1, 10)
axes[1, 1].plot(time_sampled, msd, 'bo', markersize=4, label='Simulated MSD')
axes[1, 1].plot(time_sampled, time_sampled, 'r--', linewidth=2, label='Theoretical: t')
axes[1, 1].set_xlabel('Time (steps)')
axes[1, 1].set_ylabel('Mean Squared Displacement')
axes[1, 1].set_title('Mean Squared Displacement: $\langle S_t^2 \\rangle = t$')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final position statistics ({n_simulations} simulations):")
print(f"Mean: {np.mean(final_positions):.2f} (theoretical: 0)")
print(f"Std Dev: {np.std(final_positions):.2f} (theoretical: {np.sqrt(n_steps):.2f})")

## 8. Discrete Event Simulation (DES)

**Discrete Event Simulation** models systems where state changes occur at discrete points in time.

**Example:** Queue simulation (customers arriving and being served)

**Uses calculus concepts:**
- Arrival times often follow Poisson process (uses exponential distribution)
- Service times often exponential
- Little's Law: $L = \lambda W$ (relates queue length to arrival rate and wait time)

In [ ]:
class QueueSimulation:
    def __init__(self, arrival_rate, service_rate, sim_time):
        """Simple M/M/1 queue simulation.
        
        arrival_rate: λ (arrivals per unit time)
        service_rate: μ (services per unit time)
        sim_time: total simulation time
        """
        self.arrival_rate = arrival_rate
        self.service_rate = service_rate
        self.sim_time = sim_time
        self.reset()
    
    def reset(self):
        self.time = 0
        self.queue_length = 0
        self.server_busy = False
        self.next_arrival = np.random.exponential(1/self.arrival_rate)
        self.next_departure = float('inf')
        
        # Statistics
        self.arrival_times = []
        self.departure_times = []
        self.wait_times = []
        self.queue_length_history = []
        self.time_history = []
    
    def run(self):
        """Run the simulation."""
        while self.time < self.sim_time:
            # Record state
            self.time_history.append(self.time)
            self.queue_length_history.append(self.queue_length)
            
            # Determine next event
            if self.next_arrival < self.next_departure:
                # Arrival event
                self.time = self.next_arrival
                self.arrival_times.append(self.time)
                
                if not self.server_busy:
                    # Server idle, start service immediately
                    self.server_busy = True
                    service_time = np.random.exponential(1/self.service_rate)
                    self.next_departure = self.time + service_time
                    self.wait_times.append(0)  # No wait
                else:
                    # Server busy, join queue
                    self.queue_length += 1
                
                # Schedule next arrival
                self.next_arrival = self.time + np.random.exponential(1/self.arrival_rate)
            else:
                # Departure event
                self.time = self.next_departure
                self.departure_times.append(self.time)
                
                if self.queue_length > 0:
                    # Serve next customer from queue
                    self.queue_length -= 1
                    service_time = np.random.exponential(1/self.service_rate)
                    self.next_departure = self.time + service_time
                    # Estimate wait time (simplified)
                else:
                    # No one in queue, server becomes idle
                    self.server_busy = False
                    self.next_departure = float('inf')

# Run simulation
arrival_rate = 0.8  # λ = 0.8 customers per unit time
service_rate = 1.0  # μ = 1.0 customers per unit time
sim_time = 1000

sim = QueueSimulation(arrival_rate, service_rate, sim_time)
sim.run()

# Theoretical results for M/M/1 queue
rho = arrival_rate / service_rate  # Utilization
L_theory = rho / (1 - rho)  # Average queue length
W_theory = 1 / (service_rate - arrival_rate)  # Average time in system

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Queue length over time
axes[0, 0].plot(sim.time_history, sim.queue_length_history, linewidth=1)
axes[0, 0].axhline(L_theory, color='red', linestyle='--', linewidth=2, label=f'Theoretical mean: {L_theory:.2f}')
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Queue Length')
axes[0, 0].set_title(f'Queue Length Over Time (ρ={rho:.2f})')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Queue length distribution
axes[0, 1].hist(sim.queue_length_history, bins=range(max(sim.queue_length_history)+2), 
                density=True, alpha=0.7, edgecolor='black')
# Theoretical geometric distribution: P(n) = (1-ρ)ρ^n
n_vals = np.arange(0, max(sim.queue_length_history)+1)
theoretical_pmf = (1 - rho) * (rho ** n_vals)
axes[0, 1].plot(n_vals, theoretical_pmf, 'ro-', linewidth=2, markersize=6, label='Theoretical')
axes[0, 1].set_xlabel('Queue Length')
axes[0, 1].set_ylabel('Probability')
axes[0, 1].set_title('Queue Length Distribution')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Inter-arrival times
if len(sim.arrival_times) > 1:
    inter_arrivals = np.diff(sim.arrival_times)
    axes[1, 0].hist(inter_arrivals, bins=30, density=True, alpha=0.7, edgecolor='black')
    x_arr = np.linspace(0, max(inter_arrivals), 100)
    axes[1, 0].plot(x_arr, arrival_rate * np.exp(-arrival_rate * x_arr), 'r-', 
                    linewidth=2, label=f'Exponential(λ={arrival_rate})')
    axes[1, 0].set_xlabel('Inter-arrival Time')
    axes[1, 0].set_ylabel('Density')
    axes[1, 0].set_title('Inter-arrival Time Distribution')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

# Summary statistics
avg_queue = np.mean(sim.queue_length_history)
max_queue = max(sim.queue_length_history)
n_arrivals = len(sim.arrival_times)
n_departures = len(sim.departure_times)

summary_text = f"""Simulation Results:

Parameters:
  Arrival rate (λ): {arrival_rate:.2f}
  Service rate (μ): {service_rate:.2f}
  Utilization (ρ): {rho:.2f}

Simulated:
  Total arrivals: {n_arrivals}
  Total departures: {n_departures}
  Avg queue length: {avg_queue:.2f}
  Max queue length: {max_queue}

Theoretical (M/M/1):
  Avg queue length: {L_theory:.2f}
  Avg time in system: {W_theory:.2f}
"""

axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
                verticalalignment='center', transform=axes[1, 1].transAxes)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print(summary_text)

## 9. Simulating Stochastic Differential Equations

**Geometric Brownian Motion** (used in finance for stock prices):
$$dS_t = \mu S_t dt + \sigma S_t dW_t$$

where $W_t$ is a Wiener process (Brownian motion).

**Discrete approximation (Euler-Maruyama method):**
$$S_{t+\Delta t} = S_t + \mu S_t \Delta t + \sigma S_t \sqrt{\Delta t} Z$$
where $Z \sim N(0,1)$

**Connection to calculus:** This IS calculus! Stochastic calculus with Itô's lemma.

In [ ]:
def geometric_brownian_motion(S0, mu, sigma, T, dt, n_paths=1):
    """Simulate Geometric Brownian Motion paths.
    
    S0: initial stock price
    mu: drift (expected return)
    sigma: volatility
    T: time horizon
    dt: time step
    n_paths: number of paths to simulate
    """
    n_steps = int(T / dt)
    t = np.linspace(0, T, n_steps)
    
    # Initialize
    S = np.zeros((n_steps, n_paths))
    S[0] = S0
    
    # Simulate paths
    for i in range(1, n_steps):
        Z = np.random.standard_normal(n_paths)
        S[i] = S[i-1] * (1 + mu * dt + sigma * np.sqrt(dt) * Z)
    
    return t, S

# Parameters
S0 = 100  # Initial stock price
mu = 0.1  # 10% annual drift
sigma = 0.2  # 20% annual volatility
T = 1.0  # 1 year
dt = 0.01  # Daily steps
n_paths = 100

t, S = geometric_brownian_motion(S0, mu, sigma, T, dt, n_paths)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sample paths
for i in range(min(20, n_paths)):
    axes[0, 0].plot(t, S[:, i], alpha=0.5, linewidth=1)
axes[0, 0].plot(t, S[:, :min(20, n_paths)].mean(axis=1), 'r-', linewidth=2, label='Mean path')
axes[0, 0].set_xlabel('Time (years)')
axes[0, 0].set_ylabel('Stock Price')
axes[0, 0].set_title(f'Geometric Brownian Motion Paths\n$S_0={S0}$, $\mu={mu}$, $\sigma={sigma}$')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Distribution of final prices
final_prices = S[-1, :]
axes[0, 1].hist(final_prices, bins=30, density=True, alpha=0.7, edgecolor='black')

# Theoretical: S_T is lognormal with mean and variance
# E[S_T] = S0 * exp(μT)
# Var[log(S_T)] = σ²T
mean_log = np.log(S0) + (mu - 0.5*sigma**2)*T
std_log = sigma * np.sqrt(T)
x_theory = np.linspace(final_prices.min(), final_prices.max(), 100)
pdf_theory = stats.lognorm.pdf(x_theory, s=std_log, scale=np.exp(mean_log))
axes[0, 1].plot(x_theory, pdf_theory, 'r-', linewidth=2, label='Theoretical (lognormal)')
axes[0, 1].set_xlabel('Final Stock Price')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title(f'Distribution of Final Price at T={T}')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Log returns
log_returns = np.diff(np.log(S), axis=0)
all_returns = log_returns.flatten()
axes[1, 0].hist(all_returns, bins=50, density=True, alpha=0.7, edgecolor='black')
x_ret = np.linspace(all_returns.min(), all_returns.max(), 100)
axes[1, 0].plot(x_ret, stats.norm.pdf(x_ret, mu*dt, sigma*np.sqrt(dt)), 'r-', 
                linewidth=2, label='Normal(μdt, σ√dt)')
axes[1, 0].set_xlabel('Log Return')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Distribution of Log Returns')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Mean and confidence bands
mean_path = S.mean(axis=1)
std_path = S.std(axis=1)
axes[1, 1].plot(t, mean_path, 'b-', linewidth=2, label='Mean')
axes[1, 1].fill_between(t, mean_path - 2*std_path, mean_path + 2*std_path, 
                         alpha=0.3, label='±2 std dev')
axes[1, 1].plot(t, S0 * np.exp(mu * t), 'r--', linewidth=2, label='Theoretical mean')
axes[1, 1].set_xlabel('Time (years)')
axes[1, 1].set_ylabel('Stock Price')
axes[1, 1].set_title('Mean Path with Confidence Bands')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Simulation Results ({n_paths} paths):")
print(f"Mean final price: ${final_prices.mean():.2f}")
print(f"Theoretical mean: ${S0 * np.exp(mu * T):.2f}")
print(f"Std dev final price: ${final_prices.std():.2f}")
print(f"Min final price: ${final_prices.min():.2f}")
print(f"Max final price: ${final_prices.max():.2f}")

## 10. Practice Problems

Test your understanding of simulation methods!

### Problem 1: Monte Carlo Integration

Estimate $\int_0^{\pi} \sin(x)dx$ using Monte Carlo integration with 100,000 samples.

**Hint:** The analytical answer is 2.

In [ ]:
# Your solution here
def f_sin(x):
    return np.sin(x)

n = 100000
x_samples = np.random.uniform(0, np.pi, n)
estimate = np.pi * np.mean(f_sin(x_samples))

print(f"Monte Carlo estimate: {estimate:.6f}")
print(f"Analytical answer: 2.000000")
print(f"Error: {abs(estimate - 2):.6f}")

### Problem 2: Bootstrap Confidence Interval

Given the data below, use bootstrap (10,000 resamples) to find a 95% confidence interval for the **variance**.

In [ ]:
# Data
data = np.array([12.3, 15.1, 14.5, 13.9, 16.2, 14.8, 15.5, 13.2, 14.1, 15.9])

# Your solution here
n_bootstrap = 10000
bootstrap_vars = []

for _ in range(n_bootstrap):
    sample = np.random.choice(data, size=len(data), replace=True)
    bootstrap_vars.append(np.var(sample, ddof=1))  # ddof=1 for sample variance

bootstrap_vars = np.array(bootstrap_vars)
ci = np.percentile(bootstrap_vars, [2.5, 97.5])

print(f"Original sample variance: {np.var(data, ddof=1):.4f}")
print(f"Bootstrap 95% CI for variance: [{ci[0]:.4f}, {ci[1]:.4f}]")

plt.hist(bootstrap_vars, bins=50, density=True, alpha=0.7, edgecolor='black')
plt.axvline(ci[0], color='red', linestyle='--', linewidth=2, label=f'95% CI')
plt.axvline(ci[1], color='red', linestyle='--', linewidth=2)
plt.xlabel('Variance')
plt.ylabel('Density')
plt.title('Bootstrap Distribution of Variance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Problem 3: Random Walk Return Probability

For a 1D random walk starting at 0, estimate the probability of returning to 0 within 100 steps.

Run 10,000 simulations.

In [ ]:
# Your solution here
n_steps = 100
n_simulations = 10000
returns_to_zero = 0

for _ in range(n_simulations):
    steps = np.random.choice([-1, 1], size=n_steps)
    position = np.cumsum(steps)
    
    # Check if ever returns to 0 after starting
    if np.any(position == 0):
        returns_to_zero += 1

probability = returns_to_zero / n_simulations

print(f"Estimated probability of returning to 0 within {n_steps} steps: {probability:.4f}")
print(f"Based on {n_simulations} simulations")
print(f"Number of successful returns: {returns_to_zero}")

## Summary

**Key Takeaways:**

1. **Monte Carlo Integration** approximates $\int_a^b f(x)dx$ numerically using random sampling
   - Mean value method: $(b-a) \cdot \frac{1}{N}\sum_{i=1}^N f(x_i)$
   - Converges at rate $O(1/\sqrt{N})$ regardless of dimension!

2. **Bootstrap** estimates distributions of statistics by resampling
   - Relies on CLT (calculus foundations!)
   - Provides confidence intervals without theoretical distributions

3. **Random Walks** connect to continuous processes (Brownian motion)
   - Foundation of stochastic calculus
   - Mean squared displacement grows linearly: $\langle S_t^2 \rangle = t$

4. **Discrete Event Simulation** models time-evolving systems
   - Queue theory uses exponential distributions (integration!)
   - Little's Law: $L = \lambda W$

5. **Stochastic DEs** extend calculus to random processes
   - Geometric Brownian Motion models stock prices
   - Uses Itô calculus (advanced calculus!)

**When to use simulation:**
- High-dimensional integrals
- No closed-form solution
- Validating theoretical results
- Complex stochastic systems

**Connection to calculus:** Simulation often approximates integrals, derivatives, and stochastic processes that are analytically intractable!